# Parsing TYNDP data for SHRECC

In [13]:
import pandas as pd
import numpy as np
import pickle
import os
import plotly.express as px

%load_ext jupyter_black

The jupyter_black extension is already loaded. To reload it, use:
  %reload_ext jupyter_black


# Import TYNDP data

In [14]:
filepath_xlsb = "../data/tyndp/MMStandardOutputFile_DE2050_Plexos_CY2009_v11_SoS.xlsb"
filepath_prod_pkl = "../data/tyndp/DE2050_CY2009_prod.pkl"
filepath_trade_pkl = "../data/tyndp/DE2050_CY2009_trade.pkl"
filepath_concordance = "../data/tyndp/tyndp_ei_mapping.xlsx"

In [15]:
if os.path.isfile(filepath_prod_pkl):
    with open(filepath_prod_pkl, "rb") as f:
        prod_DE2050_CY2009 = pickle.load(f)
else:
    prod_DE2050_CY2009 = pd.read_excel(
        io=filepath_xlsb,
        sheet_name="Hourly Market Data emarket",
        header=[10, 11, 12],
        index_col=[0, 1],
        engine="pyxlsb",
    )
    with open(filepath_prod_pkl, "wb") as f:
        pickle.dump(prod_DE2050_CY2009, f)

if os.path.isfile(filepath_trade_pkl):
    with open(filepath_trade_pkl, "rb") as f:
        trade_DE2050_CY2009 = pickle.load(f)
else:
    trade_DE2050_CY2009 = pd.read_excel(
        io=filepath_xlsb,
        sheet_name="Crossborder exchanges",
        header=[10],
        index_col=[0, 1],
        engine="pyxlsb",
    )
    with open(filepath_trade_pkl, "wb") as f:
        pickle.dump(trade_DE2050_CY2009, f)

In [16]:
# This is the raw data from TYNDP
# Parsing and cleaning includes:
# - keep only the columns with production data for nodes XX00, XX00RETE
# - remove columns where all values are NaN
# - set a proper datetime index

prod_DE2050_CY2009.head()

,Category,Nuclear [MW],Lignite old 1 [MW],Lignite old 2 [MW],Lignite new [MW],Lignite CCS [MW],Hard coal old 1 [MW],Hard coal old 2 [MW],Hard coal new [MW],Hard coal CCS [MW],Gas conventional old 1 [MW],...,Hydrogen CCGT [MW],Demand Side Response [MW],CH4 Heat Pump (load) [MW],H2 Heat Pump (load) [MW],Adequacy [MW],Demand [MW]\n(losses included),Balance [MW],Dumped Energy [MW],Energy Not Served [MW],Marginal Cost [€]
,Country,AL00,AL00,AL00,AL00,AL00,AL00,AL00,AL00,AL00,AL00,...,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE
,Date /Code,AL00_1,AL00_2,AL00_3,AL00_4,AL00_5,AL00_6,AL00_7,AL00_8,AL00_9,AL00_10,...,UKNIRETE_52,UKNIRETE_DSR,UKNIRETE_HCH4,UKNIRETE_HH2,UKNIRETE_SoS,UKNIRETE_LOAD,UKNIRETE_Balance,UKNIRETE_Dump,UKNIRETE_ENS,UKNIRETE_Mgl Cost
1,01JAN00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,2214,-2214,0,0,237
2,01JAN01:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,2206,-2206,0,0,233
3,01JAN02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,2177,-2177,0,0,231
4,01JAN03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,2173,-2173,0,0,231
5,01JAN04:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,2150,-2150,0,0,231


In [17]:
# index in the proper format
dt = pd.to_datetime(
    prod_DE2050_CY2009.index.get_level_values(1) + "2050", format="%d%b%H:%M%Y"
)

## Production

In [18]:
prod_DE2050_CY2009.dropna(axis=1, how="all", inplace=True)
prod_DE2050_CY2009.index = dt
prod_DE2050_CY2009.head()

Category            Gas CCGT old 2 [MW] Gas CCGT new [MW] Run-of-River [MW]  \
Country                            AL00              AL00              AL00   
Date /Code                      AL00_13           AL00_14           AL00_26   
2050-01-01 00:00:00                   0               110               193   
2050-01-01 01:00:00                   0               100               193   
2050-01-01 02:00:00                   0                90               193   
2050-01-01 03:00:00                   0                 0               193   
2050-01-01 04:00:00                   0                 0               193   

Category            Reservoir [MW] Wind Onshore [MW]  \
Country                       AL00              AL00   
Date /Code                 AL00_27           AL00_31   
2050-01-01 00:00:00           1904                40   
2050-01-01 01:00:00           1904                43   
2050-01-01 02:00:00           1904                48   
2050-01-01 03:00:00           1904                55   
2050-01-01 04:00:00           1904                59   

Category            Solar (Photovoltaic) [MW] Demand [MW]\n(losses included)  \
Country                                  AL00                           AL00   
Date /Code                            AL00_33                      AL00_LOAD   
2050-01-01 00:00:00                         0                           1145   
2050-01-01 01:00:00                         0                           1006   
2050-01-01 02:00:00                         0                            923   
2050-01-01 03:00:00                         0                            885   
2050-01-01 04:00:00                         0                            892   

Category            Balance [MW] Dumped Energy [MW] Energy Not Served [MW]  \
Country                     AL00               AL00                   AL00   
Date /Code          AL00_Balance          AL00_Dump               AL00_ENS   
2050-01-01 00:00:00         1102                  0                      0   
2050-01-01 01:00:00         1234                  0                      0   
2050-01-01 02:00:00         1312                  0                      0   
2050-01-01 03:00:00         1267                  0                      0   
2050-01-01 04:00:00         1265                  0                      0   

Category             ...                Marginal Cost [€]  \
Country              ...     UKNI SRES          UKNI SRES   
Date /Code           ... UKNI SRES_ENS UKNI SRES_Mgl Cost   
2050-01-01 00:00:00  ...             0              -1000   
2050-01-01 01:00:00  ...             0              -1000   
2050-01-01 02:00:00  ...             0              -1000   
2050-01-01 03:00:00  ...             0              -1000   
2050-01-01 04:00:00  ...             0              -1000   

Category            Solar (Photovoltaic) [MW] CH4 Heat Pump (load) [MW]  \
Country                              UKNIRETE                  UKNIRETE   
Date /Code                        UKNIRETE_33             UKNIRETE_HCH4   
2050-01-01 00:00:00                         0                         0   
2050-01-01 01:00:00                         0                         0   
2050-01-01 02:00:00                         0                         0   
2050-01-01 03:00:00                         0                         0   
2050-01-01 04:00:00                         0                         0   

Category            H2 Heat Pump (load) [MW] Demand [MW]\n(losses included)  \
Country                             UKNIRETE                       UKNIRETE   
Date /Code                      UKNIRETE_HH2                  UKNIRETE_LOAD   
2050-01-01 00:00:00                        0                           2214   
2050-01-01 01:00:00                        0                           2206   
2050-01-01 02:00:00                        0                           2177   
2050-01-01 03:00:00                        0                           2173   
2050-01-0

## Trade

Trade data requires special attention. Nodes are treated separately (this is OK), but it looks like various interconnection alternatives exist (Concept 1/Concept 2), this is unclear.

In [19]:
# same for trade data
trade_DE2050_CY2009.dropna(axis=1, how="all", inplace=True)
trade_DE2050_CY2009.index = dt
trade_DE2050_CY2009.head()

,AL00-GR00,AL00-GR00 Concept 1,AL00-GR00 Concept 2,AL00-ME00,AL00-ME00 Concept 1,AL00-ME00 Concept 2,AL00-ME00 Concept 3,AL00-ME00 Concept 4,AL00-MK00,AL00-MK00 Concept 1,...,UKNIEV-UKNIRETE,UKNIRETE-UKNIEV,UKNI-UKNIEV,UKNI-UKNIRETE,UKOH001OHEL-UK00 EXP,UKOH003OHEL-UK00 EXP,UKOH004OHEL-IE00 EXP,UKOH004OHEL-UK00 EXP,UKOH005OHEL-UK00 EXP,UKOH006OHEL-UK00 EXP
2050-01-01 00:00:00,-600,-500,-500,330,468,468,468,424,-330,-330,...,0,0,0,2214,-3877,-2377,-2687,5610,8276,6526
2050-01-01 01:00:00,-600,-500,-500,298,439,439,439,383,-185,-185,...,0,0,0,2206,-3877,-4437,-2574,5610,8261,6397
2050-01-01 02:00:00,-600,-500,-500,314,456,456,456,405,-191,-191,...,0,2,1,2179,-3877,-4844,-2489,5610,8201,6139
2050-01-01 03:00:00,-600,-500,-500,323,463,463,463,416,-234,-234,...,0,2,1,2175,-3877,-4627,-2291,5610,8177,6053
2050-01-01 04:00:00,-600,-500,-500,329,467,467,467,422,-247,-247,...,0,1,1,2152,-3877,-4459,-2093,5610,8226,5967


In [ ]:
# We want to reshape the trade data with origin-destination pairs as multiindex columns
parsed = trade_DE2050_CY2009.columns.to_series().str.extract(
    r"^\s*(?P<origin>.+?)\s*-\s*(?P<destination>.+?)\s*$"
)

# If any origin/destination is NaN, raise a flag
bad_cols = trade_DE2050_CY2009.columns[
    parsed["origin"].isna() | parsed["destination"].isna()
]
print(bad_cols)

# Clean the names by removing trailing "Concept N" and stripping whitespace
parsed["origin"] = (
    parsed["origin"].str.replace(r"\s*/?Concept\s+\d+\s*$", "", regex=True).str.strip()
)

parsed["destination"] = (
    parsed["destination"]
    .str.replace(r"\s*/?Concept\s+\d+\s*$", "", regex=True)
    .str.strip()
)

# Apply the cleaned names to the columns
trade = trade_DE2050_CY2009.copy()
trade.columns = pd.MultiIndex.from_arrays(
    [parsed["origin"], parsed["destination"]], names=["origin", "destination"]
)

# Origin-destination node pairs for each hour
od_hourly = trade.T.groupby(level=["origin", "destination"]).sum().T

Index([], dtype='object')


In [21]:
od_hourly.head()

origin               AL00                    AT00                             \
destination          GR00  ME00 MK00  RS00 AT00EV AT00RETE  CH00  CZ00  DE00   
2050-01-01 00:00:00 -1600  2158 -705  1250      0     7605 -1700  2400  7147   
2050-01-01 01:00:00 -1600  1998 -414  1250      0     7465 -1700  2400  6801   
2050-01-01 02:00:00 -1600  2087 -426  1250      0     7358 -1700  2400  6510   
2050-01-01 03:00:00 -1600  2128 -512  1250      0     7266 -1700  2400  6702   
2050-01-01 04:00:00 -1600  2152 -538  1250      0     7192 -1700  2400  6902   

origin                     ...     UKNI UKNIEV          UKNIRETE UKOH001OHEL  \
destination          HU00  ... UKNIRETE   UKNI UKNIRETE   UKNIEV    UK00 EXP   
2050-01-01 00:00:00 -1124  ...     2214      0        0        0       -3877   
2050-01-01 01:00:00 -1262  ...     2206      0        0        0       -3877   
2050-01-01 02:00:00 -1336  ...     2179      0        0        2       -3877   
2050-01-01 03:00:00 -1343  ...     2175      0        0        2       -3877   
2050-01-01 04:00:00 -1258  ...     2152      0        0        1       -3877   

origin              UKOH003OHEL UKOH004OHEL          UKOH005OHEL UKOH006OHEL  
destination            UK00 EXP    IE00 EXP UK00 EXP    UK00 EXP    UK00 EXP  
2050-01-01 00:00:00       -2377       -2687     5610        8276        6526  
2050-01-01 01:00:00       -4437       -2574     5610        8261        6397  
2050-01-01 02:00:00       -4844       -2489     5610        8201        6139  
2050-01-01 03:00:00       -4627       -2291     5610        8177        6053  
2050-01-01 04:00:00       -4459       -2093     5610        8226        5967  

[5 rows x 388 columns]

In [22]:
lu_imports = od_hourly.columns.get_level_values("destination").str.startswith("LU")
lu_exports = od_hourly.columns.get_level_values("origin").str.startswith("LU")


print(od_hourly.loc[:, lu_imports].head())
print(od_hourly.loc[:, lu_exports].head())

origin              BE00                  DE00      FR00   LUG1           \
destination         LUB1 LUG1 LUG1 Real 1 LUG1 LUV1 LUF1 LUG1EV LUG1RETE   
2050-01-01 00:00:00   27  800         500  374    0  121    228     1098   
2050-01-01 01:00:00   25  800         500  332    0  134    228     1126   
2050-01-01 02:00:00   26  800         500  316    0  125    227     1150   
2050-01-01 03:00:00   25  800         500  321    0  130    222     1142   
2050-01-01 04:00:00   25  800         500  313    0  113    218     1129   

origin              LUG1EV          LUG1RETE  
destination           LUG1 LUG1RETE   LUG1EV  
2050-01-01 00:00:00      0        0      327  
2050-01-01 01:00:00      0        0      367  
2050-01-01 02:00:00      0        0      387  
2050-01-01 03:00:00      0        0      375  
2050-01-01 04:00:00      0        0      351  
origin                LUG1          LUG1EV          LUG1RETE
destination         LUG1EV LUG1RETE   LUG1 LUG1RETE   LUG1EV
2050-01-01 00:00

In [35]:
od_hourly.loc[:, lu_imports].sum()

origin    destination
BE00      LUB1            248394
          LUG1           1926318
          LUG1 Real 1    1051391
DE00      LUG1           6219713
          LUV1            343015
FR00      LUF1           1470076
LUG1      LUG1EV          898530
          LUG1RETE       4658159
LUG1EV    LUG1              5738
          LUG1RETE         11500
LUG1RETE  LUG1EV         2088340
dtype: int64

In [33]:
px.area(
    od_hourly.loc[:, lu_imports]
    .resample("ME")
    .sum()
    .groupby(level="origin", axis=1)
    .sum()
)

C:\Users\Gibon\AppData\Local\Temp\ipykernel_16040\2336434019.py:5: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  .groupby(level="origin", axis=1)


In [46]:
country = "LU"

prod_DE2050_CY2009_agg = prod_DE2050_CY2009.groupby(level="Country", axis=1).sum()

country_columns = prod_DE2050_CY2009_agg.columns.str.startswith(country)
prod_DE2050_CY2009_agg.loc[:, country_columns].resample("ME").sum()

C:\Users\Gibon\AppData\Local\Temp\ipykernel_16040\741722360.py:3: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  prod_DE2050_CY2009_agg = prod_DE2050_CY2009.groupby(level="Country", axis=1).sum()


Country,LUB1,LUB1 SRES,LUF1,LUF1 SRES,LUG1,LUG1 EV Passenger Prosumer,LUG1 EV Passenger Street,LUG1 SRES,LUG1RETE,LUV1,LUV1 SRES
2050-01-31,77592,-744000,61023,-744000,627468,384573,180196,-744000,350604,136721,-744000
2050-02-28,59731,-672000,48624,-672000,565597,346748,146082,-672000,286283,166223,-672000
2050-03-31,35694,-744000,32122,-744000,705745,307259,139951,-744000,424646,175900,-744000
2050-04-30,27900,-720000,26352,-720000,561681,293927,120613,-720000,522638,291743,-720000
2050-05-31,19784,-744000,22491,-744000,562022,285612,102438,-744000,523064,259431,-744000
2050-06-30,25455,-720000,22833,-720000,535257,285036,113238,-720000,547829,297605,-720000
2050-07-31,24066,-744000,23130,-744000,568206,284918,105733,-744000,562797,295928,-744000
2050-08-31,24985,-744000,25362,-744000,467980,312565,117083,-744000,568552,332852,-744000
2050-09-30,27042,-720000,28108,-720000,474197,282222,112429,-720000,423378,123305,-720000
2050-10-31,42130,-744000,38692,-744000,549008,351134,149471,-744000,346453,119872,-744000


In [8]:
sources = prod_DE2050_CY2009.columns.levels[0]

In [9]:
concordance = pd.read_excel(
    filepath_concordance, sheet_name="concordance", index_col=0, skipfooter=1
).iloc[:, :-1]
concordance = concordance.loc[concordance.sum(1) > 0, concordance.sum() > 0]
concordance /= concordance.sum()
concordance.columns.name = "Category"
concordance.head()

Category,Gas CCGT CCS [MW],Gas CCGT new [MW],Gas CCGT old 1 [MW],Gas CCGT old 2 [MW],Gas CCGT present 1 [MW],Gas CCGT present 2 [MW],Gas OCGT new [MW],Gas OCGT old [MW],Gas conventional old 1 [MW],Gas conventional old 2 [MW],...,Others non-renewable [MW],Others renewable [MW],Pump Storage - Closed Loop (turbine) [MW],Pump Storage - Open Loop (turbine) [MW],Reservoir [MW],Run-of-River [MW],Solar (Photovoltaic) [MW],Solar (Thermal) [MW],Wind Offshore [MW],Wind Onshore [MW]
premise activity,,,,,,,,,,,,,,,,,,,,,
"electricity production, at biomass-fired IGCC power plant, pre, pipeline 200km, storage 1000m",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"electricity production, at co-generation natural gas-fired power plant, post, pipeline 200km, storage 1000m",0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"electricity production, at co-generation oil-fired power plant, post, pipeline 200km, storage 1000m",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"electricity production, at co-generation wood-fired power plant, post, pipeline 200km, storage 1000m",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"electricity production, at hard coal-fired IGCC power plant",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
country = "LU"
index_c = [c for c in prod_DE2050_CY2009.columns.levels[1] if c.startswith(country)]
index_c

['LUB1',
 'LUB1 SRES',
 'LUF1',
 'LUF1 SRES',
 'LUG1',
 'LUG1 EV Passenger Prosumer',
 'LUG1 EV Passenger Street',
 'LUG1 SRES',
 'LUG1RETE',
 'LUV1',
 'LUV1 SRES']

In [11]:
idx = pd.IndexSlice

prod_DE2050_CY2009.loc[:, idx[:, index_c]].dropna(how="all", axis=1).sum().sort_values()

Category                        Country                     Date /Code                         
Marginal Cost [€]               LUV1 SRES                   LUV1 SRES_Mgl Cost                    -8736000
                                LUF1 SRES                   LUF1 SRES_Mgl Cost                    -8736000
                                LUB1 SRES                   LUB1 SRES_Mgl Cost                    -8736000
                                LUG1 SRES                   LUG1 SRES_Mgl Cost                    -8736000
Balance [MW]                    LUG1                        LUG1_Balance                          -3647483
                                                                                                    ...   
Solar (Photovoltaic) [MW]       LUG1RETE                    LUG1RETE_33                            1711841
Demand [MW]\n(losses included)  LUG1 EV Passenger Prosumer  LUG1 EV Passenger Prosumer_LOAD        1958164
Marginal Cost [€]               LUG1 EV Passenge

In [12]:
month = 1

sample = (
    prod_DE2050_CY2009.loc[:, idx[concordance.columns, index_c]]
    .dropna(how="all", axis=1)
    .iloc[month * 30 * 24 : (month * 30 + 7) * 24, :]
    .droplevel(level=0, axis=0)
    .groupby("Category", axis=1)
    .sum()
)

sample = sample.loc[:, sample.sum() > 0]


# ax = sample.plot.area(figsize=(10, 6))

C:\Users\Gibon\AppData\Local\Temp\ipykernel_19076\477859567.py:8: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  .groupby("Category", axis=1)


In [13]:
sample.index = pd.to_datetime(sample.index, format="%d%b%H:%M").map(
    lambda x: x.replace(year=2050)
)

In [14]:
plot_sample = sample.iloc[:, :10] if sample.shape[1] > 10 else sample
fig = go.Figure()
for col in plot_sample.columns:
    fig.add_trace(
        go.Scatter(
            x=plot_sample.index,
            y=plot_sample[col],
            name=str(col),
            mode="lines",
            stackgroup="one",
        )
    )
fig.update_layout(
    title="TYNDP sample generation",
    xaxis_title="Time",
    yaxis_title="Generation (MW)",
    hovermode="x unified",
    height=600,
    width=1000,
)
fig.show()

# Match with premise-ecoinvent inventories

The strategy is to recover the supply mixes for each country in a premise database.

In [18]:
cols = sample.columns.intersection(concordance.columns)
A = concordance.loc[:, cols]
B = sample.loc[:, cols]

In [23]:
A.shape

(38, 6)

In [24]:
B.shape

(168, 6)

In [25]:
A @ B.T

: 

In [36]:
print("A", A.shape, A.dtypes.unique())
print("B", B.shape, B.dtypes.unique())
print("cols", len(cols))
print("A unique", A.columns.is_unique)
print("B unique", B.columns.is_unique)
print("same cols", A.columns.equals(B.columns))
print("estimated output MB", B.shape[0] * A.shape[0] * 8 / 1e6)

A (38, 5) [dtype('float64')]
B (168, 5) [dtype('int64')]
cols 5
A unique True
B unique True
same cols True
estimated output MB 0.051072


In [38]:
A_np = A.to_numpy(dtype="float64", copy=True)
B_np = B.to_numpy(dtype="float64", copy=True)

print(A_np.shape, A_np.flags)
print(B_np.shape, B_np.flags)
print(np.isfinite(A_np).all(), np.isfinite(B_np).all())

(38, 5)   C_CONTIGUOUS : False
  F_CONTIGUOUS : True
  OWNDATA : False
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

(168, 5)   C_CONTIGUOUS : False
  F_CONTIGUOUS : True
  OWNDATA : False
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

True True


In [39]:
out = np.einsum("tk,ak->ta", B_np, A_np)

In [41]:
np.show_config()

Build Dependencies:
  blas:
    detection method: pkgconfig
    found: true
    include directory: D:/.conda/envs/shrecc/Library/include
    lib directory: D:/.conda/envs/shrecc/Library/lib
    name: blas
    openblas configuration: unknown
    pc file directory: D:\bld\numpy_1707225519803\_h_env\Library\lib\pkgconfig
    version: 3.9.0
  lapack:
    detection method: internal
    found: true
    include directory: unknown
    lib directory: unknown
    name: dep3117830679024
    openblas configuration: unknown
    pc file directory: unknown
    version: 1.26.4
Compilers:
  c:
    commands: cl.exe
    linker: link
    name: msvc
    version: 19.29.30153
  c++:
    commands: cl.exe
    linker: link
    name: msvc
    version: 19.29.30153
  cython:
    commands: cython
    linker: cython
    name: cython
    version: 3.0.8
Machine Information:
  build:
    cpu: x86_64
    endian: little
    family: x86_64
    system: windows
  host:
    cpu: x86_64
    endian: little
    family: x86_64
 

In [24]:
B_np @ A_np.T

: 

In [26]:
sample.T

2050-05-01 00:00:00  \
Category                                  Country  Date /Code                         
Others renewable [MW]                     LUG1     LUG1_35                      123   
Pump Storage - Closed Loop (turbine) [MW] LUV1     LUV1_29                        0   
Run-of-River [MW]                         LUG1     LUG1_26                       12   
Solar (Photovoltaic) [MW]                 LUG1     LUG1_33                        0   
                                          LUG1RETE LUG1RETE_33                    0   
Wind Onshore [MW]                         LUG1     LUG1_31                       14   

                                                                2050-05-01 01:00:00  \
Category                                  Country  Date /Code                         
Others renewable [MW]                     LUG1     LUG1_35                      123   
Pump Storage - Closed Loop (turbine) [MW] LUV1     LUV1_29                        0   
Run-of-River [MW]                         LUG1     LUG1_26                       12   
Solar (Photovoltaic) [MW]                 LUG1     LUG1_33                        0   
                                          LUG1RETE LUG1RETE_33                    0   
Wind Onshore [MW]                         LUG1     LUG1_31                        0   

                                                                2050-05-01 02:00:00  \
Category                                  Country  Date /Code                         
Others renewable [MW]                     LUG1     LUG1_35                      126   
Pump Storage - Closed Loop (turbine) [MW] LUV1     LUV1_29                        0   
Run-of-River [MW]                         LUG1     LUG1_26                       12   
Solar (Photovoltaic) [MW]                 LUG1     LUG1_33                        0   
                                          LUG1RETE LUG1RETE_33                    0   
Wind Onshore [MW]                         LUG1     LUG1_31                       10   

                                                                2050-05-01 03:00:00  \
Category                                  Country  Date /Code                         
Others renewable [MW]                     LUG1     LUG1_35                      126   
Pump Storage - Closed Loop (turbine) [MW] LUV1     LUV1_29                        0   
Run-of-River [MW]                         LUG1     LUG1_26                       12   
Solar (Photovoltaic) [MW]                 LUG1     LUG1_33                        0   
                                          LUG1RETE LUG1RETE_33                    0   
Wind Onshore [MW]                         LUG1     LUG1_31                       21   

                                                                2050-05-01 04:00:00  \
Category                                  Country  Date /Code                         
Others renewable [MW]                     LUG1     LUG1_35                      127   
Pump Storage - Closed Loop (turbine) [MW] LUV1     LUV1_29                        0   
Run-of-River [MW]                         LUG1     LUG1_26                       12   
Solar (Photovoltaic) [MW]                 LUG1     LUG1_33                        0   
                                          LUG1RETE LUG1RETE_33                    0   
Wind Onshore [MW]                         LUG1     LUG1_31                       19   

                                                                2050-05-01 05:00:00  \
Category                                  Country  Date /Code                         
Others renewable [MW]                     LUG1     LUG1_35                      129   
Pump Storage - Closed Loop (turbine) [MW] LUV1     LUV1_29                        0   
Run-of-River [MW]                         LUG1     LUG1_26                       12   
Solar (Photovoltaic) [MW]                 LUG1     LUG1_33                        8   
                                          LUG1RETE LUG1RE